# Задача 4. Распределения заряда и энергетическая зависимость

Для каждой энергии $E=0{,}2;\,0{,}5;\,0{,}7;\,1;\,1{,}5;\,2;\,3$ МэВ
моделируем 1000 событий в центре, затем 1000 событий из точки $(0,0,1)$ м.
Определяем средний заряд, стандартное отклонение и относительную ширину
$\delta_Q=\sigma_Q/\langle Q\rangle$.

Реализация серий находится в [charge_study.py](charge_study.py).
Каждое направление фотона и каждый отклик на попадание разыгрываются явно.

In [ ]:
import csv
import json
from pathlib import Path
import charge_study
from IPython.display import Image, Markdown, display
%matplotlib inline

# Полный повторный расчёт: 14 серий по 1000 событий.
# Данные сохраняются относительно charge_study.py, в папку charge_results.
charge_study.main()

z=0 м, E=0.2 МэВ: mean=24.9260, sigma=6.0867, width=24.419%
z=0 м, E=0.5 МэВ: mean=62.7920, sigma=9.6507, width=15.369%
z=0 м, E=0.7 МэВ: mean=86.8880, sigma=11.3882, width=13.107%
z=0 м, E=1 МэВ: mean=124.6360, sigma=13.9995, width=11.232%
z=0 м, E=1.5 МэВ: mean=187.4330, sigma=16.8636, width=8.997%
z=0 м, E=2 МэВ: mean=250.0580, sigma=18.7020, width=7.479%
z=0 м, E=3 МэВ: mean=374.5570, sigma=24.7118, width=6.598%
z=1 м, E=0.2 МэВ: mean=24.8060, sigma=5.9111, width=23.829%
z=1 м, E=0.5 МэВ: mean=62.3030, sigma=9.7522, width=15.653%
z=1 м, E=0.7 МэВ: mean=86.4160, sigma=11.4237, width=13.219%
z=1 м, E=1 МэВ: mean=125.2530, sigma=13.7844, width=11.005%
z=1 м, E=1.5 МэВ: mean=188.1150, sigma=15.9716, width=8.490%
z=1 м, E=2 МэВ: mean=249.8920, sigma=19.3635, width=7.749%


In [ ]:
folder = Path(charge_study.__file__).resolve().parent / 'charge_results'
results = json.loads((folder/'study_results.json').read_text(encoding='utf-8'))
records = results['statistics']
print('Всего серий:', len(records))
print('Всего событий:', sum(row['n_events'] for row in records))
table = ['| E, МэВ | Среднее, центр | Ширина, центр | Среднее, z=1 м | Ширина, z=1 м |',
         '|---:|---:|---:|---:|---:|']
for E in results['energies_MeV']:
    r0 = next(r for r in records if r['E_MeV']==E and r['z_m']==0)
    r1 = next(r for r in records if r['E_MeV']==E and r['z_m']==1)
    table.append(f"| {E:g} | {r0['mean_Q_pe']:.2f} | {r0['relative_width_percent']:.2f}% | "
                 f"{r1['mean_Q_pe']:.2f} | {r1['relative_width_percent']:.2f}% |")
display(Markdown('\n'.join(table)))

In [ ]:
display(Image(filename=str(folder/'charge_distributions.png')))

В центре $p=Nr^2/(4R^2)=0{,}025$. Из формул задачи 3:
$$\frac{\sigma_Q}{\langle Q\rangle}
=\sqrt{\frac{1+a(1-p)}{YEpa}}
\simeq\frac{10{,}91\%}{\sqrt{E/(1\,\text{МэВ})}}.$$
Средний заряд растёт пропорционально $E$, абсолютная ширина — как
$\sqrt E$, относительная ширина — как $E^{-1/2}$.

In [ ]:
display(Image(filename=str(folder/'charge_trends.png')))
print('Показатели степенной аппроксимации относительной ширины:')
for z, exponent in results['width_power_law_exponents'].items():
    print(f'z = {z} м: {exponent:.6f}')

In [ ]:
r0 = next(r for r in records if r['E_MeV']==1 and r['z_m']==0)
r1 = next(r for r in records if r['E_MeV']==1 and r['z_m']==1)
delta = r1['mean_Q_pe']-r0['mean_Q_pe']
error = (r0['se_mean_Q_pe']**2+r1['se_mean_Q_pe']**2)**0.5
print('События 1 МэВ; ошибки средних:')
print(f"Центр: {r0['mean_Q_pe']:.3f} ± {r0['se_mean_Q_pe']:.3f}")
print(f"z=1 м: {r1['mean_Q_pe']:.3f} ± {r1['se_mean_Q_pe']:.3f}")
print(f'Разность: {delta:.3f} ± {error:.3f} фотоэлектронов')
print('Независимый геометрический расчёт изменения среднего, %:',
      results['relative_mean_shift_percent'])

После смещения ближайшие ФЭУ получают больше света, дальние — меньше.
При почти равномерном покрытии изменения почти компенсируются в сумме.
Для этой геометрии интегрирование телесных углов даёт изменение среднего
на $-0{,}101\%$. В выборках по 1000 событий статистически значимого
изменения среднего заряда не обнаружено.

Все 14 000 событий записаны в `charge_results/event_charges.csv`.
Сводная таблица находится в `charge_results/charge_summary.csv`.
На графиках погрешности соответствуют оценкам $1\sigma$.